# Training-only pressure-level probabilities for v007

Definitions only. Execute the code cells from `research/sequence_components_v005.ipynb` first, then these cells in the same namespace. Import/definition performs no dataset reads, fitting, or training; all v005 public definitions remain unchanged.

The recurrent encoder, head hidden width/SiLU, preprocessing, optimizer parameters, shuffling, and LR machinery follow v005. The explicit change is the scalar output and masked L1 objective becoming K categorical logits with **inspiratory categorical cross entropy**. Decode the **lower posterior median on native pressure levels**, the MAE-optimal action for the learned posterior. Cross entropy is an ordinary categorical proper loss; it does not directly penalize ordinal distances. No regression, residual baseline, target normalization, pressure-grid inference, label smoothing, or nearest-grid snapping is used.

The codebook contains sorted unique float32 pressure values from **all phases of only the supplied training fitter**. On A, supply Afit only, never all labeled data. Expiratory pressure values broaden support but do not contribute CE. A/B labels are used only for native MAE and a reported unseen-level coverage diagnostic. Unseen validation levels remain legal metric values; no validation categorical loss is calculated. An unseen training label causes an explicit failure. Native pressure values use the same float32 conversion as v005.

Pass the existing compact BiLSTM `model_spec` unchanged, the combined47 feature names, the exact frozen200 cosine rates, seed `20261102`, and the same A grouped split. The caller must copy the existing train config and explicitly change `loss` to `categorical_ce`; the component rejects a silently inherited L1 objective. This notebook neither chooses those artifacts nor fits assignment data. `v7_fit_validate` keeps best-A plus a separate authoritative `.last` checkpoint; `v7_fit_fixed` accepts no validation arrays and requires a frozen rate per budgeted epoch. Reports are JSON serializable and epoch logs contain native inspiratory MAE plus training categorical CE.

Resumption fingerprints the model/config/seed, raw fitter and validation arrays, feature names, fitted feature statistics, codebook, and runtime. Checkpoints retain model, Adam moments, scheduler, Python/NumPy/Torch/CUDA RNGs, and DataLoader shuffle generator. Resume requires the same implementation, numeric runtime and device type; the caller should additionally lock notebook source hashes. Best and last checkpoints contain trusted-local Python/Torch serialization and must not be loaded from untrusted sources.

The K-way head is applied **after selecting inspiration** during training; the full input sequence still enters the recurrent encoder. Inference decodes head chunks of 2048 positions, independent of breath batch size, so it need not allocate `[batch,80,K]`. For roughly 950 classes and batch 256 this reduces head activations, but actual CUDA memory/throughput must be profiled on the target 11 GB GPU; CPU toy verification does not establish GPU capacity. `prediction_batch_size=128` is a conservative caller choice and does not alter training.

Optional `v7_profile_pressure_cuda(model_spec, train_cfg=train_cfg)` allocates only synthetic arrays at the requested 47-feature/950-class/batch256 shape, runs one masked-CE backward/Adam update plus chunked median inference, and reports device memory peaks and first-update timings. It never runs on definition and reads no assignment data. It preserves caller RNG/backend settings; run in an otherwise idle process for meaningful capacity measurements.

Optional `v7_pressure_resume_selfcheck(device="cuda", work_dir=None)` uses only seven synthetic breaths and tests native inference, exact model/Adam/scheduler/RNG/shuffle state restoration, A-style plateau selection, and fixed-rate replay at epoch boundaries. It returns JSON-safe proof and deletes its temporary files. Run it before production on the target runtime. The recorded CPU checks separately cover fresh Python processes.

Strict deterministic CUDA decoding keeps softmax on the GPU, computes cumulative probabilities and the median index on CPU (required by Torch 2.2 CUDA cumsum restrictions), then returns predictions to the original device. CE and its gradients stay on CUDA. Inference uses the existing 2048-point head chunks; online training metrics transfer only the masked inspiration logits. Determinism is never disabled or weakened.


In [ ]:
# Imports, explicit objective, and the unchanged recurrent encoder
"""Run after the v005 engine cells. Definitions only: no data reads or training."""
_V7_REQUIRED = ('V5PressureRNN', 'canonical_model_spec', '_train_configuration',
    'fit_preprocessor', 'transform_inputs', '_training_arrays', '_sequence_device',
    '_optimizer', '_rng_state', '_restore_rng', '_atomic_torch_save', '_array_sha256',
    '_last_path', '_json_safe', '_epoch_event', '_file_sha256', '_cpu_copy')
if any(name not in globals() for name in _V7_REQUIRED):
    raise RuntimeError('Execute research/sequence_components_v005.ipynb engine definitions first')
V7_PRESSURE_VERSION = 'v007-pressure-categorical-ce-v2'
V7_CODEBOOK_PHASE = 'all_training_phases'
V7_HEAD_CHUNK_POINTS = 2048


class V7PressureCodebookRNN(V5PressureRNN):
    """v005 recurrent encoder/head hidden layer; K pressure logits replace scalar output.

    `mask` selects positions BEFORE applying the probability head. The full offline
    input breath is still encoded, preserving bidirectional context. No R/C heads,
    pressure residuals, ordinal surrogate, smoothing, or regression loss is implied.
    """
    def __init__(self, model_spec, n_features, n_levels):
        super().__init__(model_spec, n_features)
        if self.spec['rc_heads']:
            raise ValueError('v007 probability head requires rc_heads=False')
        self.n_levels = int(n_levels)
        if self.n_levels < 1:
            raise ValueError('At least one training pressure level is required')
        old_output = self.head[-1]
        self.head[-1] = nn.Linear(old_output.in_features, self.n_levels)

    def encode(self, x):
        # Matches V5PressureRNN.forward through its recurrent encoder exactly.
        if x.ndim != 3 or x.shape[-1] != self.n_features+1:
            raise ValueError('Expected standardized numeric features plus internal R/C code')
        joint = x[..., -1].long()
        sequence = x[..., :self.n_features]
        if self.joint_embedding is not None:
            sequence = torch.cat((sequence, self.joint_embedding(joint)), dim=-1)
        if not self.explicit_stack:
            sequence, _ = self.rnn(sequence)
        else:
            outputs = []
            for i, layer in enumerate(self.recurrent_layers):
                previous = sequence
                sequence, _ = layer(previous)
                if self.spec['residual']:
                    sequence = sequence + self.skip_projections[i](previous)
                outputs.append(sequence)
                if i+1 < len(self.recurrent_layers):
                    sequence = self.between_layers(sequence)
            if self.spec['output_concat']:
                sequence = torch.cat(outputs, dim=-1)
        return sequence

    def forward(self, x, mask=None):
        encoded = self.encode(x)
        if mask is not None:
            if mask.dtype != torch.bool or mask.shape != x.shape[:2]:
                raise ValueError('Probability-head mask must be boolean [breaths,80]')
            encoded = encoded[mask]
        return self.head(encoded)


def v7_build_pressure_model(model_spec, n_features, n_levels):
    return V7PressureCodebookRNN(model_spec, n_features, n_levels)


def v7_pressure_parameter_count(model_spec, n_features, n_levels):
    return int(sum(p.numel() for p in v7_build_pressure_model(
        model_spec, n_features, n_levels).parameters()))


def _v7_train_configuration(train_cfg):
    supplied = copy.deepcopy(dict(train_cfg))
    if supplied.get('loss') != 'categorical_ce':
        raise ValueError("v007 requires explicit loss='categorical_ce'; do not silently reuse v005 L1")
    supplied['loss'] = 'l1'  # Reuse v005 validation of every unchanged train field.
    cfg = _train_configuration(supplied)
    if cfg['residual_baseline']:
        raise ValueError('Pressure levels are native; residual_baseline must be False')
    cfg['loss'] = 'categorical_ce'
    return cfg


def v7_deterministic_lr_schedule(train_cfg):
    cfg = _v7_train_configuration(train_cfg)
    return deterministic_lr_schedule(dict(cfg, loss='l1'))




In [ ]:
# Training-only pressure codebook, exact labels, and posterior median

def v7_fit_pressure_codebook(y_train, u_out_train):
    """Use ONLY the supplied fitter's float32 labels, including expiration.

    All training phases broaden support without consulting A/B/test. Only
    inspiration contributes CE. Train label values are represented exactly after
    the same float32 conversion as v005. No grid is inferred from held-out labels.
    """
    y, out = np.asarray(y_train, dtype=np.float32), np.asarray(u_out_train)
    if y.ndim != 2 or y.shape[1] != V5_STEPS or not len(y) or out.shape != y.shape:
        raise ValueError('Expected nonempty fitter y/u_out[breaths,80]')
    if not np.isfinite(y).all() or not np.isin(out, (0, 1)).all() or not (out == 0).any():
        raise ValueError('Finite fitter targets and nonempty binary inspiration mask are required')
    levels = np.unique(y).astype(np.float32, copy=False)
    return dict(levels=levels, levels_sha256=_array_sha256(levels),
        phase=V7_CODEBOOK_PHASE, value_dtype='float32',
        fitted_training_breaths=int(len(y)), fitted_training_points=int(y.size),
        fitted_inspiratory_points=int((out == 0).sum()),
        label_policy='exact_training_values; held-out unseen levels are native-MAE-only')


def _v7_validated_codebook(codebook):
    result = copy.deepcopy(codebook)
    levels = np.array(result['levels'], dtype=np.float32, copy=True)
    if levels.ndim != 1 or not len(levels) or not np.isfinite(levels).all():
        raise ValueError('Invalid pressure codebook')
    if not np.all(levels[1:] > levels[:-1]):
        raise ValueError('Pressure codebook must be strictly increasing')
    if result.get('phase') != V7_CODEBOOK_PHASE or result.get('value_dtype') != 'float32':
        raise ValueError('Pressure codebook policy mismatch')
    if result.get('levels_sha256') != _array_sha256(levels):
        raise ValueError('Pressure codebook fingerprint mismatch')
    result['levels'] = levels
    return result


def _v7_level_indices(y, codebook):
    values = np.asarray(y, dtype=np.float32)
    if not np.isfinite(values).all():
        raise ValueError('Nonfinite pressure label')
    levels = _v7_validated_codebook(codebook)['levels']
    index = np.searchsorted(levels, values)
    safe = np.minimum(index, len(levels)-1)
    known = (index < len(levels)) & (levels[safe] == values)
    return index.astype(np.int64), known


def v7_encode_training_pressure(y, codebook):
    """Exact categorical mapping; reject unseen training labels rather than snap."""
    index, known = _v7_level_indices(y, codebook)
    if not known.all():
        raise ValueError(f'{int((~known).sum())} training labels outside fitter-only pressure codebook')
    return np.ascontiguousarray(index)


def v7_codebook_coverage(y, u_out, codebook):
    """Diagnostic only; held-out labels never mutate the codebook or train loss."""
    values, out = np.asarray(y, dtype=np.float32), np.asarray(u_out)
    if values.shape != out.shape or values.ndim != 2 or values.shape[1] != V5_STEPS:
        raise ValueError('Coverage requires matching pressure/mask[breaths,80]')
    if not np.isin(out, (0, 1)).all() or not (out == 0).any():
        raise ValueError('Coverage needs a nonempty binary inspiration mask')
    _, known = _v7_level_indices(values, codebook)
    unknown_values = np.unique(values[~known])
    return dict(total_points=int(values.size), unseen_points=int((~known).sum()),
        scored_points=int((out == 0).sum()),
        unseen_scored_points=int(((~known) & (out == 0)).sum()),
        unseen_distinct_levels=int(len(unknown_values)),
        unseen_levels_preview=unknown_values[:20].tolist(),
        held_out_policy='evaluate native MAE without class-mapping or snapping')


def v7_decode_pressure_median(logits, levels):
    """Lower posterior median on sorted native levels, the MAE-optimal action.

    At an exact 0.5 boundary, choose the lower level. A tiny rounding error that
    pushes every cumulative value below 0.5 is safely clamped to the final level.
    """
    levels = torch.as_tensor(levels, device=logits.device, dtype=torch.float32)
    if logits.shape[-1] != len(levels) or logits.shape[-1] < 1:
        raise ValueError('Pressure logits/codebook dimension mismatch')
    probabilities = logits.float().softmax(dim=-1)
    # Torch 2.2 rejects CUDA cumsum under strict deterministic algorithms.
    # Decode only (never CE/backprop): transfer probabilities to CPU for the
    # cumulative reduction, then return median indices to the original device.
    # Inference already limits this transfer to V7_HEAD_CHUNK_POINTS rows.
    if logits.device.type == 'cuda' and torch.are_deterministic_algorithms_enabled():
        cumulative = probabilities.detach().cpu().cumsum(dim=-1)
        index = (cumulative < .5).sum(dim=-1).clamp_max(len(levels)-1)
        index = index.to(device=logits.device)
    else:
        cumulative = probabilities.cumsum(dim=-1)
        index = (cumulative < .5).sum(dim=-1).clamp_max(len(levels)-1)
    return levels[index]


@torch.no_grad()
def _v7_predict_native(model, transformed, codebook, device, batch_size):
    batch_size = int(batch_size)
    if batch_size < 1:
        raise ValueError('Prediction batch_size must be positive')
    model.eval()
    output = np.empty(transformed.shape[:2], dtype=np.float32)
    levels = torch.as_tensor(codebook['levels'], device=device, dtype=torch.float32)
    for start in range(0, len(transformed), batch_size):
        xb = torch.from_numpy(transformed[start:start+batch_size]).to(device)
        encoded = model.encode(xb)
        flat = encoded.reshape(-1, encoded.shape[-1])
        prediction = torch.empty(len(flat), device=device, dtype=torch.float32)
        for position in range(0, len(flat), V7_HEAD_CHUNK_POINTS):
            logits = model.head(flat[position:position+V7_HEAD_CHUNK_POINTS])
            prediction[position:position+V7_HEAD_CHUNK_POINTS] = v7_decode_pressure_median(logits, levels)
        output[start:start+batch_size] = prediction.reshape(xb.shape[:2]).cpu().numpy()
    if not np.isfinite(output).all():
        raise RuntimeError('Nonfinite native pressure median')
    return output




In [ ]:
# Fingerprints, loader, masked CE epochs, and trusted checkpoint loading

def _v7_json_sha256(value):
    return hashlib.sha256(json.dumps(_json_safe(value), sort_keys=True,
        separators=(',', ':'), allow_nan=False).encode()).hexdigest()


def _v7_preprocessing_sha256(preprocess):
    return _v7_json_sha256(_validated_preprocessor(preprocess))


def _v7_runtime(device):
    import platform
    return dict(torch_version=str(torch.__version__), numpy_version=str(np.__version__),
        python_version=platform.python_version(), machine=platform.machine(),
        device_type=device.type, cuda_version=torch.version.cuda,
        cudnn_version=torch.backends.cudnn.version(),
        cuda_device_count=torch.cuda.device_count() if device.type == 'cuda' else 0,
        cuda_name=torch.cuda.get_device_name(device) if device.type == 'cuda' else None,
        deterministic=True, torch_threads=4, head_chunk_points=V7_HEAD_CHUNK_POINTS)


def _v7_fingerprint(raw, y, out, spec, cfg, seed, feature_names, mode,
                    prep, codebook, device, extra):
    result = _fingerprint(raw, y, out, spec, cfg, seed, feature_names, mode, extra)
    result.update(implementation=V7_PRESSURE_VERSION, runtime=_v7_runtime(device),
        preprocessing_sha256=_v7_preprocessing_sha256(prep),
        codebook_sha256=codebook['levels_sha256'],
        codebook_policy=V7_CODEBOOK_PHASE, objective='inspiratory_categorical_cross_entropy',
        decoder='lower_native_pressure_posterior_median')
    return result


def _v7_loader(raw, y, out, prep, codebook, cfg, seed, device):
    generator = torch.Generator().manual_seed(int(seed))
    transformed = transform_inputs(raw, prep)
    data = TensorDataset(torch.from_numpy(transformed),
        torch.from_numpy(v7_encode_training_pressure(y, codebook)),
        torch.from_numpy(np.ascontiguousarray(out == 0)))
    loader = DataLoader(data, batch_size=cfg['batch_size'], shuffle=True, num_workers=0,
        pin_memory=device.type == 'cuda', generator=generator)
    return loader, generator, transformed


def _v7_training_epoch(model, loader, optimizer, cfg, codebook, device):
    model.train()
    total_error, total_ce, count, steps = 0., 0., 0, 0
    levels = torch.as_tensor(codebook['levels'], device=device, dtype=torch.float32)
    for xb, yb, mask in loader:
        xb, yb, mask = (v.to(device, non_blocking=device.type == 'cuda') for v in (xb, yb, mask))
        if not bool(mask.any()):
            continue
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb, mask=mask)  # No [B,80,K] training logits for expiration.
        target = yb[mask]
        loss = F.cross_entropy(logits, target, reduction='mean')
        if not torch.isfinite(loss):
            raise RuntimeError('Nonfinite categorical CE')
        with torch.no_grad():
            median = v7_decode_pressure_median(logits.detach(), levels)
            total_error += float((median-levels[target]).abs().double().sum())
            total_ce += float(loss.detach())*target.numel()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), float(cfg['gradient_clip']))
        optimizer.step()
        count += target.numel()
        steps += 1
    if not count:
        raise ValueError('Training epoch contained no inspiratory points')
    return dict(training_online_inspiratory_mae=total_error/count,
        training_online_categorical_ce=total_ce/count, training_scored_points=count,
        optimizer_steps=steps)


def _v7_payload(model, prep, codebook, spec, cfg, seed, fingerprint, optimizer,
                scheduler, generator, epoch, history, completed=False, termination=None):
    payload = _checkpoint_payload(model, prep, spec, cfg, seed, fingerprint, optimizer,
        scheduler, generator, epoch, history, completed, termination)
    payload.update(format_version=7, component=V7_PRESSURE_VERSION,
        pressure_codebook=_cpu_copy(codebook),
        preprocessing_sha256=_v7_preprocessing_sha256(prep),
        objective='inspiratory_categorical_cross_entropy',
        decoder='lower_native_pressure_posterior_median',
        runtime=_v7_runtime(next(model.parameters()).device))
    return payload


def v7_load_pressure_checkpoint(checkpoint_path, device='cpu'):
    """Load trusted local v007 codebook checkpoints for native median prediction."""
    payload = torch.load(Path(checkpoint_path), map_location='cpu', weights_only=False)
    if payload.get('format_version') != 7 or payload.get('component') != V7_PRESSURE_VERSION:
        raise ValueError('Expected matching v007 pressure-codebook checkpoint')
    prep = _validated_preprocessor(payload['preprocess'])
    codebook = _v7_validated_codebook(payload['pressure_codebook'])
    if payload.get('preprocessing_sha256') != _v7_preprocessing_sha256(prep):
        raise ValueError('Checkpoint feature-statistics fingerprint mismatch')
    if payload['fingerprint']['codebook_sha256'] != codebook['levels_sha256']:
        raise ValueError('Checkpoint pressure codebook fingerprint differs from stage recipe')
    if payload['fingerprint']['preprocessing_sha256'] != payload['preprocessing_sha256']:
        raise ValueError('Checkpoint feature statistics differ from stage recipe')
    model = v7_build_pressure_model(payload['model_spec'], prep['input_shape'][1],
        len(codebook['levels'])).to(_sequence_device(device))
    model.load_state_dict(payload['state_dict'], strict=True)
    model.eval()
    return model, prep, codebook, payload


def v7_predict_checkpoint(checkpoint_path, x, device='cpu', batch_size=128):
    model, prep, codebook, _ = v7_load_pressure_checkpoint(checkpoint_path, device)
    return _v7_predict_native(model, transform_inputs(x, prep), codebook,
        _sequence_device(device), batch_size)




In [ ]:
# Validation-selected A fitting and fixed-budget fitting with exact boundary resume

def _v7_fit(x_train, y_train, u_out_train, model_spec, train_cfg, seed, checkpoint_path,
            epoch_log_path, feature_names, device, resume, on_epoch, validation,
            train_diagnostic_breaths, lr_schedule):
    began = time.monotonic()
    device = _sequence_device(device)
    spec, cfg = canonical_model_spec(model_spec), _v7_train_configuration(train_cfg)
    if spec['rc_heads']:
        raise ValueError('v007 pressure head requires rc_heads=False')
    raw, y, out = _training_arrays(x_train, y_train, u_out_train)
    mode = 'pressure_validation' if validation is not None else 'pressure_fixed'
    if validation is not None:
        val_raw, val_y, val_out = _training_arrays(*validation)
        if val_raw.shape[1:] != raw.shape[1:]:
            raise ValueError('Training and validation feature dimensions differ')
        if int(train_diagnostic_breaths) < 1:
            raise ValueError('train_diagnostic_breaths must be positive')
        rates = v7_deterministic_lr_schedule(cfg)
        budget = cfg['max_epochs']
        extra = dict(val_x_sha256=_array_sha256(val_raw), val_y_sha256=_array_sha256(val_y),
            val_out_sha256=_array_sha256(val_out), train_diagnostic_breaths=int(train_diagnostic_breaths))
    else:
        rates = [float(rate) for rate in lr_schedule]
        if not rates or not all(np.isfinite(rate) and rate >= 0 for rate in rates):
            raise ValueError('Fixed training requires a nonempty finite nonnegative LR sequence')
        if len(rates) != cfg['max_epochs']:
            raise ValueError('Fixed LR schedule length must equal train_cfg max_epochs')
        budget = len(rates)
        extra = dict(lr_schedule=rates)
    path, last = Path(checkpoint_path), _last_path(checkpoint_path)
    if not resume and (path.exists() or last.exists()):
        raise FileExistsError('Use a fresh path or explicitly resume the existing stage')
    # Recompute exclusively from this fitter's arrays, including at resume, then
    # verify exact statistics and codebook against saved stage fingerprints.
    prep = fit_preprocessor(raw, y, out, spec, feature_names,
        scaler_phase=cfg['scaler_phase'], residual_baseline=False)
    codebook = v7_fit_pressure_codebook(y, out)
    fingerprint = _v7_fingerprint(raw, y, out, spec, cfg, seed, feature_names, mode,
        prep, codebook, device, extra)
    seed_sequence(seed)
    model = v7_build_pressure_model(spec, raw.shape[-1], len(codebook['levels'])).to(device)
    optimizer = _optimizer(model, cfg)
    loader, generator, train_z = _v7_loader(raw, y, out, prep, codebook, cfg, seed, device)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
        factor=cfg['scheduler_factor'], patience=cfg['scheduler_patience'],
        threshold=cfg['scheduler_threshold'], threshold_mode='rel', min_lr=cfg['min_lr']) \
        if validation is not None and cfg['scheduler'] == 'plateau' else None
    if validation is not None:
        val_z = transform_inputs(val_raw, prep)
        coverage = v7_codebook_coverage(val_y, val_out, codebook)
        diagnostic_ids = np.sort(np.random.default_rng(20261110).choice(len(raw),
            size=min(int(train_diagnostic_breaths), len(raw)), replace=False))
    else:
        coverage, diagnostic_ids = None, None
    history, best, bad_epochs, first_epoch, elapsed_before = [], None, 0, 1, 0.
    completed, termination = False, 'epoch_cap' if validation is not None else 'fixed_A_budget'
    if resume:
        if not last.is_file():
            raise FileNotFoundError(f'Missing authoritative last-state checkpoint: {last}')
        saved = torch.load(last, map_location='cpu', weights_only=False)
        if (saved.get('format_version') != 7 or saved.get('component') != V7_PRESSURE_VERSION
                or saved.get('fingerprint') != fingerprint):
            raise ValueError('Resume rejected: data, recipe, seed, runtime, statistics, or codebook changed')
        saved_prep = _validated_preprocessor(saved['preprocess'])
        saved_codebook = _v7_validated_codebook(saved['pressure_codebook'])
        if (_v7_preprocessing_sha256(saved_prep) != fingerprint['preprocessing_sha256']
                or saved_codebook['levels_sha256'] != codebook['levels_sha256']):
            raise ValueError('Resume rejected: saved preprocessing/codebook was altered')
        model.load_state_dict(saved['state_dict'], strict=True)
        optimizer.load_state_dict(saved['optimizer_state'])
        if scheduler is not None:
            scheduler.load_state_dict(saved['scheduler_state'])
        _restore_rng(saved['rng_state'], generator)
        history, first_epoch = copy.deepcopy(saved['epochs']), saved['epoch']+1
        best, bad_epochs = saved.get('best_payload'), saved.get('bad_epochs', 0)
        elapsed_before = float(saved.get('elapsed_seconds', 0.))
        completed, termination = saved['completed'], saved['termination']
    else:
        initial = _v7_payload(model, prep, codebook, spec, cfg, seed, fingerprint, optimizer,
            scheduler, generator, 0, [], completed=False, termination='initialized')
        initial.update(best_payload=None, bad_epochs=0, elapsed_seconds=0.,
            validation_used=validation is not None)
        _atomic_torch_save(initial, last)
    _epoch_event(epoch_log_path, dict(event='resume' if resume else 'start',
        component=V7_PRESSURE_VERSION, first_epoch=first_epoch, model_spec=spec, train_cfg=cfg,
        deterministic_lr_schedule=rates, codebook_levels=len(codebook['levels']),
        codebook_sha256=codebook['levels_sha256'], codebook_phase=V7_CODEBOOK_PHASE,
        preprocessing_sha256=fingerprint['preprocessing_sha256'],
        held_out_coverage=coverage, validation_used=validation is not None))
    if not completed:
        for epoch in range(first_epoch, budget+1):
            epoch_began = time.monotonic()
            if rates is not None:
                for group in optimizer.param_groups:
                    group['lr'] = rates[epoch-1]
            lr_used = float(optimizer.param_groups[0]['lr'])
            training = _v7_training_epoch(model, loader, optimizer, cfg, codebook, device)
            improved = False
            if validation is not None:
                val_mae = inspiratory_mae(val_y, _v7_predict_native(model, val_z, codebook,
                    device, cfg['prediction_batch_size']), val_out)
                train_mae = inspiratory_mae(y[diagnostic_ids], _v7_predict_native(model,
                    train_z[diagnostic_ids], codebook, device, cfg['prediction_batch_size']), out[diagnostic_ids])
                best_mae = float('inf') if best is None else best['best_val_mae']
                improved = val_mae < best_mae-float(cfg['min_delta'])
                bad_epochs = 0 if improved else bad_epochs+1
                if scheduler is not None:
                    scheduler.step(val_mae)
            elapsed = elapsed_before+time.monotonic()-began
            record = dict(event='epoch', epoch=epoch, lr_used=lr_used,
                lr_next=(rates[epoch] if epoch < len(rates) else None) if rates is not None
                    else float(optimizer.param_groups[0]['lr']),
                **training, cumulative_optimizer_steps=sum(r['optimizer_steps'] for r in history)
                    +training['optimizer_steps'],
                epoch_seconds=time.monotonic()-epoch_began, elapsed_seconds=elapsed,
                validation_used=validation is not None)
            if validation is not None:
                record.update(validation_inspiratory_mae=val_mae,
                    training_eval_inspiratory_mae=train_mae,
                    training_diagnostic_breaths=len(diagnostic_ids),
                    best_val_mae=val_mae if improved else best_mae,
                    best_epoch=epoch if improved else best['best_epoch'])
            history.append(record)
            completed = epoch == budget
            termination = 'epoch_cap' if validation is not None else 'fixed_A_budget'
            if validation is not None and cfg['patience'] and bad_epochs >= cfg['patience']:
                completed, termination = True, 'early_stopping'
            if not completed and cfg['max_seconds'] is not None and time.monotonic()-began >= cfg['max_seconds']:
                termination = 'time_cap'
            current = _v7_payload(model, prep, codebook, spec, cfg, seed, fingerprint, optimizer,
                scheduler, generator, epoch, history, completed, termination)
            current.update(bad_epochs=bad_epochs, elapsed_seconds=elapsed,
                validation_used=validation is not None, held_out_coverage=coverage)
            if validation is not None:
                if improved:
                    best = copy.deepcopy(current)
                    best.update(best_epoch=epoch, best_val_mae=val_mae,
                        training_eval_mae=train_mae, selected_lr_schedule=[r['lr_used'] for r in history])
                current['best_payload'] = best
            else:
                current.update(selected_lr_schedule=rates, completed_epochs=epoch)
            _atomic_torch_save(current, last)
            if validation is None or improved:
                _atomic_torch_save(best if validation is not None else current, path)
            _epoch_event(epoch_log_path, record)
            if on_epoch is not None:
                on_epoch(copy.deepcopy(record))
            if completed or termination == 'time_cap':
                break
    if validation is not None:
        if best is None:
            raise RuntimeError('No validation epoch has completed')
        model.load_state_dict(best['state_dict'], strict=True)
        restored = inspiratory_mae(val_y, _v7_predict_native(model, val_z, codebook,
            device, cfg['prediction_batch_size']), val_out)
        if not np.isclose(restored, best['best_val_mae'], rtol=1e-6, atol=1e-6):
            raise RuntimeError('Restored pressure median checkpoint metric differs')
        best.update(executed_epochs=len(history), executed_lr_schedule=[r['lr_used'] for r in history],
            completed=bool(completed), termination=termination,
            stage_completed=bool(completed), stage_termination=termination)
        _atomic_torch_save(best, path)
        selected_rates = best['selected_lr_schedule']
    else:
        # Restore .last to public path even after interruption between its writes.
        _atomic_torch_save(torch.load(last, map_location='cpu', weights_only=False), path)
        selected_rates = rates
    report = dict(component=V7_PRESSURE_VERSION, model_spec=spec, train_cfg=cfg,
        seed=int(seed), epochs=history, selected_lr_schedule=selected_rates,
        deterministic_lr_schedule=rates, executed_lr_schedule=[r['lr_used'] for r in history],
        executed_epochs=len(history), completed_epochs=len(history),
        completed=bool(completed), termination=termination,
        optimizer_steps=sum(r['optimizer_steps'] for r in history),
        validation_used=validation is not None, training_breaths=len(raw),
        elapsed_seconds=elapsed_before+time.monotonic()-began,
        parameter_count=int(sum(p.numel() for p in model.parameters())),
        checkpoint_path=str(path), checkpoint_sha256=_file_sha256(path),
        last_checkpoint_path=str(last), preprocessing=_json_safe(prep),
        preprocessing_sha256=fingerprint['preprocessing_sha256'],
        pressure_codebook=_json_safe(codebook), pressure_levels=len(codebook['levels']),
        fingerprint=fingerprint)
    if validation is not None:
        report.update(validation_mae=restored, best_epoch=best['best_epoch'],
            training_eval_mae=best['training_eval_mae'], training_diagnostic_breaths=len(diagnostic_ids),
            validation_breaths=len(val_raw), scored_validation_points=int((val_out == 0).sum()),
            held_out_coverage=coverage, best_checkpoint_restored=True)
    _epoch_event(epoch_log_path, dict(event='finished', completed=completed, termination=termination,
        checkpoint_sha256=report['checkpoint_sha256'], validation_used=validation is not None,
        validation_mae=report.get('validation_mae'), best_epoch=report.get('best_epoch')))
    return _json_safe(report)


def v7_fit_validate(x_train, y_train, u_out_train, x_val, y_val, u_out_val,
                    model_spec, train_cfg, seed, checkpoint_path, epoch_log_path=None,
                    feature_names=None, train_diagnostic_breaths=2048, device='cuda',
                    resume=False, on_epoch=None):
    """Select best native inspiratory MAE on A; fitter-only codebook and scalers.

    Caller supplies disjoint whole-breath groups. Held-out pressure levels absent
    from the fitter are reported and evaluated natively, never added or snapped.
    Resumption uses the authoritative .last state with Adam/scheduler/all RNGs.
    `on_epoch` runs only after atomic checkpoint writes and JSONL logging.
    """
    return _v7_fit(x_train, y_train, u_out_train, model_spec, train_cfg, seed,
        checkpoint_path, epoch_log_path, feature_names, device, resume, on_epoch,
        (x_val, y_val, u_out_val), train_diagnostic_breaths, None)


def v7_fit_fixed(x_train, y_train, u_out_train, model_spec, train_cfg, seed, lr_schedule,
                 checkpoint_path, epoch_log_path=None, feature_names=None, device='cuda',
                 resume=False, on_epoch=None):
    """Fit fresh codebook/scalers using only fitter data; no held-out arrays accepted.

    Replay the supplied frozen A rate sequence (length must equal max_epochs).
    The authoritative .last file permits exact same-runtime epoch-boundary resume.
    """
    return _v7_fit(x_train, y_train, u_out_train, model_spec, train_cfg, seed,
        checkpoint_path, epoch_log_path, feature_names, device, resume, on_epoch,
        None, None, lr_schedule)



In [ ]:
# Optional synthetic CUDA capacity smoke; never runs on definition

def v7_profile_pressure_cuda(model_spec, n_features=47, n_levels=950, batch_size=256,
                             inspiratory_steps=30, seed=20261102, device='cuda',
                             train_cfg=None, inference_batch_size=128):
    """One synthetic masked-CE Adam update + full-breath median inference.

    No dataset is read. Pass the actual compact spec and training config to check
    the target GPU before the authorized run. Reports first-update timings, not
    steady-state throughput. Retains Adam moments during inference to model the
    training/validation memory situation. Leaves no checkpoint or model behind.
    """
    requested = torch.device(device)
    if requested.type != 'cuda':
        raise ValueError('This optional capacity profile requires a CUDA device')
    shape = dict(n_features=int(n_features), n_levels=int(n_levels),
        batch_size=int(batch_size), inspiratory_steps=int(inspiratory_steps),
        inference_batch_size=int(inference_batch_size))
    if (shape['n_features'] < 2 or min(shape['n_levels'], shape['batch_size'],
            shape['inference_batch_size']) < 1 or not 1 <= shape['inspiratory_steps'] <= V5_STEPS):
        raise ValueError('Invalid synthetic profile dimensions')
    spec = canonical_model_spec(model_spec)
    if spec['rc_heads']:
        raise ValueError('v007 probability head requires rc_heads=False')
    cfg = _v7_train_configuration(dict(loss='categorical_ce', scheduler='constant',
        max_epochs=2) if train_cfg is None else train_cfg)
    report = dict(component=V7_PRESSURE_VERSION, synthetic_only=True,
        model_spec=spec, shape=shape, seed=int(seed), optimizer=cfg['optimizer'],
        loss=cfg['loss'], head_chunk_points=V7_HEAD_CHUNK_POINTS,
        timing_scope='first complete update, then chunked median inference; no warmup')
    if not torch.cuda.is_available():
        return dict(report, status='cuda_unavailable', success=False)
    selected = _sequence_device(requested)
    # Preserve caller random state and deterministic/backend settings.
    generator = torch.Generator()
    rng = _rng_state(generator)
    backend = dict(threads=torch.get_num_threads(),
        deterministic=torch.are_deterministic_algorithms_enabled(),
        warn_only=torch.is_deterministic_algorithms_warn_only_enabled(),
        benchmark=torch.backends.cudnn.benchmark,
        cudnn_deterministic=torch.backends.cudnn.deterministic,
        matmul_tf32=torch.backends.cuda.matmul.allow_tf32,
        cudnn_tf32=torch.backends.cudnn.allow_tf32)
    model = optimizer = xb = target = mask = logits = loss = prediction = None
    try:
        seed_sequence(seed)
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(selected)
        model = v7_build_pressure_model(spec, shape['n_features'], shape['n_levels']).to(selected)
        optimizer = _optimizer(model, cfg)
        xb = torch.randn(shape['batch_size'], V5_STEPS, shape['n_features']+1, device=selected)
        xb[..., -1] = torch.randint(0, 9, (shape['batch_size'], 1), device=selected)
        target = torch.randint(0, shape['n_levels'], xb.shape[:2], device=selected)
        mask = torch.arange(V5_STEPS, device=selected)[None, :].expand(xb.shape[:2]) < shape['inspiratory_steps']
        transformed = xb.detach().cpu().numpy()
        model.train()
        torch.cuda.synchronize(selected)
        began = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb, mask=mask)
        loss = F.cross_entropy(logits, target[mask])
        if not bool(torch.isfinite(loss)):
            raise RuntimeError('Synthetic pressure profile produced nonfinite CE')
        loss.backward()
        norm = nn.utils.clip_grad_norm_(model.parameters(), float(cfg['gradient_clip']))
        optimizer.step()
        torch.cuda.synchronize(selected)
        report.update(training_seconds=time.perf_counter()-began,
            training_peak_allocated_bytes=int(torch.cuda.max_memory_allocated(selected)),
            training_peak_reserved_bytes=int(torch.cuda.max_memory_reserved(selected)),
            training_categorical_ce=float(loss.detach()), gradient_norm_before_clip=float(norm),
            masked_head_shape=list(logits.shape),
            parameter_count=int(sum(p.numel() for p in model.parameters())))
        optimizer.zero_grad(set_to_none=True)
        xb = target = mask = logits = loss = None
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(selected)
        levels = np.linspace(-10., 100., shape['n_levels'], dtype=np.float32)
        torch.cuda.synchronize(selected)
        began = time.perf_counter()
        prediction = _v7_predict_native(model, transformed, {'levels':levels}, selected,
            shape['inference_batch_size'])
        torch.cuda.synchronize(selected)
        properties = torch.cuda.get_device_properties(selected)
        report.update(inference_seconds=time.perf_counter()-began,
            inference_peak_allocated_bytes=int(torch.cuda.max_memory_allocated(selected)),
            inference_peak_reserved_bytes=int(torch.cuda.max_memory_reserved(selected)),
            prediction_shape=list(prediction.shape), predictions_finite=bool(np.isfinite(prediction).all()),
            predictions_in_codebook=bool(np.isin(prediction, levels).all()),
            device_name=properties.name, device_total_bytes=int(properties.total_memory),
            runtime=_v7_runtime(selected), status='passed', success=True)
    except torch.cuda.OutOfMemoryError as exc:
        report.update(status='cuda_out_of_memory', success=False, error=str(exc),
            peak_allocated_bytes=int(torch.cuda.max_memory_allocated(selected)),
            peak_reserved_bytes=int(torch.cuda.max_memory_reserved(selected)))
    finally:
        model = optimizer = xb = target = mask = logits = loss = prediction = None
        torch.cuda.empty_cache()
        _restore_rng(rng, generator)
        torch.set_num_threads(backend['threads'])
        torch.use_deterministic_algorithms(backend['deterministic'], warn_only=backend['warn_only'])
        torch.backends.cudnn.benchmark = backend['benchmark']
        torch.backends.cudnn.deterministic = backend['cudnn_deterministic']
        torch.backends.cuda.matmul.allow_tf32 = backend['matmul_tf32']
        torch.backends.cudnn.allow_tf32 = backend['cudnn_tf32']
    return _json_safe(report)



In [ ]:
# Optional synthetic same-runtime epoch-boundary checkpoint proof

def v7_pressure_resume_selfcheck(device='cpu', work_dir=None):
    """Synthetic save/resume proof callable before production; never autoruns.

    Tests both A-style plateau selection and validation-free replay. CPU external
    checks additionally exercise separate Python processes; this callable tests
    exact epoch-boundary state restoration on the currently selected runtime/GPU.
    Returns JSON-safe evidence and deletes temporary toy artifacts on completion.
    """
    import tempfile
    selected = _sequence_device(device)
    rng = np.random.default_rng(827)
    x = rng.normal(size=(7, V5_STEPS, 5)).astype(np.float32)
    x[..., 0] = np.asarray([5, 20, 50, 5, 20, 50, 20])[:, None]
    x[..., 1] = np.asarray([10, 20, 50, 20, 50, 10, 10])[:, None]
    out = np.zeros((7, V5_STEPS), dtype=np.int8)
    out[:, 43:] = 1
    y = np.where(x[..., 2] < -.4, -3., np.where(x[..., 2] > .4, 3., 1.)).astype(np.float32)
    y[:, 43:] = 7.
    vx, vy, vo = x[:3].copy(), y[:3].copy(), out[:3].copy()
    vx[..., 2] += .25
    vy[0, :4] = 9.5  # Deliberately absent from training-only support.
    spec = dict(family='lstm', hidden=4, layers=2, dropout=.25,
        bidirectional=True, joint_embedding_dim=2, head_width=5)
    base = dict(lr=.003, min_lr=.0001, batch_size=3, max_epochs=3,
        patience=0, loss='categorical_ce', prediction_batch_size=2)
    checks = {}
    class _V7ProofInterrupt(Exception):
        pass
    def interrupt(record):
        if record['epoch'] == 1:
            raise _V7ProofInterrupt()
    def same(a, b):
        if torch.is_tensor(a):
            return torch.equal(a, b)
        if isinstance(a, np.ndarray):
            return np.array_equal(a, b)
        if isinstance(a, dict):
            return a.keys() == b.keys() and all(same(a[k], b[k]) for k in a)
        if isinstance(a, (list, tuple)):
            return len(a) == len(b) and all(same(v, w) for v, w in zip(a, b))
        return a == b
    def metric_rows(payload):
        return [{k:v for k, v in record.items() if k not in ('epoch_seconds', 'elapsed_seconds')}
            for record in payload['epochs']]
    with tempfile.TemporaryDirectory(prefix='v007-pressure-proof-', dir=work_dir) as temporary:
        folder = Path(temporary)
        for mode in ('validation', 'fixed'):
            rates = [.003, .002, .001]
            cfg = dict(base, scheduler='plateau', scheduler_patience=0) if mode == 'validation' \
                else dict(base, scheduler='replay', frozen_lr_schedule=rates)
            def run(path, resume=False, callback=None):
                kwargs = dict(model_spec=spec, train_cfg=cfg, seed=20261102,
                    checkpoint_path=path, feature_names=['R', 'C', 'a', 'b', 'c'],
                    device=str(selected), resume=resume, on_epoch=callback)
                if mode == 'validation':
                    return v7_fit_validate(x, y, out, vx, vy, vo,
                        train_diagnostic_breaths=5, **kwargs)
                return v7_fit_fixed(x, y, out, lr_schedule=rates, **kwargs)
            uninterrupted, resumed = folder/(mode+'-serial.pt'), folder/(mode+'-resumed.pt')
            serial_report = run(uninterrupted)
            try:
                run(resumed, callback=interrupt)
            except _V7ProofInterrupt:
                pass
            else:
                raise RuntimeError('Synthetic resume proof did not interrupt after epoch one')
            resumed_report = run(resumed, resume=True)
            a = torch.load(_last_path(uninterrupted), map_location='cpu', weights_only=False)
            b = torch.load(_last_path(resumed), map_location='cpu', weights_only=False)
            for field in ('state_dict', 'optimizer_state', 'scheduler_state', 'rng_state',
                          'preprocess', 'pressure_codebook', 'fingerprint'):
                checks[mode+'_'+field+'_exact'] = bool(same(a[field], b[field]))
            checks[mode+'_epoch_metrics_exact'] = bool(same(metric_rows(a), metric_rows(b)))
            checks[mode+'_completed_three_epochs'] = bool(a['completed'] and b['completed']
                and a['epoch'] == b['epoch'] == 3)
            pa = v7_predict_checkpoint(uninterrupted, vx, device=str(selected), batch_size=2)
            pb = v7_predict_checkpoint(resumed, vx, device=str(selected), batch_size=2)
            checks[mode+'_public_native_predictions_exact'] = bool(np.array_equal(pa, pb))
            checks[mode+'_native_predictions_in_training_support'] = bool(np.isin(pa, [-3., 1., 3., 7.]).all())
            if mode == 'validation':
                checks['validation_unseen_support_reported'] = bool(
                    serial_report['held_out_coverage']['unseen_scored_points'] == 4)
                checks['validation_bestA_mae_restored'] = bool(np.isclose(
                    inspiratory_mae(vy, pa, vo), serial_report['validation_mae'], rtol=0., atol=1e-6))
                checks['validation_plateau_state_present'] = a['scheduler_state'] is not None
            else:
                checks['fixed_no_validation_labels'] = not resumed_report['validation_used']
                checks['fixed_exact_frozen_rates'] = [r['lr_used'] for r in b['epochs']] == rates
    return dict(component=V7_PRESSURE_VERSION, synthetic_only=True, dataset_reads=0,
        runtime=_v7_runtime(selected), proof_scope='same-runtime epoch-boundary resume',
        checks=checks, checks_passed=sum(checks.values()), checks_total=len(checks),
        passed=all(checks.values()))
